# Entrega 2 / Nahuel Piñeyro / 5230736-3

## Taller 8 – Demanda de bicicletas (RNN)

### Importación

In [1]:
import numpy as np
import pandas as pd
from datetime import datetime
import calendar
import tensorflow as tf
from utils import FilledIn, TimeFeatures, create_sequence_datasets
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from tensorflow import keras

#### Preprocesamiento

In [2]:
df_train_raw = pd.read_csv('train.csv')

train = FilledIn(df_train_raw)
days = train['datetime'].dt.day
df_full_train = train[days <= 15]
df_full_val = train[days >= 16]

cat_features = ['season', 'weather', 'month', 'weekday', 'hour']
num_features = ['temp', 'atemp', 'humidity', 'windspeed']
scaler = ColumnTransformer([('cat', OneHotEncoder(), cat_features),
                            ('num', StandardScaler(), num_features),
                            ], remainder='passthrough')

preprocess_pipe = Pipeline([('timefeatures', TimeFeatures()),
                            ('scaler', scaler)])

dropeable = ['registered', 'casual', 'count']

df_x_train = df_full_train.copy()
df_x_train = df_x_train.drop(columns=dropeable)
df_x_val = df_full_val.copy()
df_x_val = df_x_val.drop(columns=dropeable)
X_train = preprocess_pipe.fit_transform(df_x_train).toarray()
X_val = preprocess_pipe.transform(df_x_val).toarray()

y_train = df_full_train[['count']]
y_train = y_train.set_index(pd.to_datetime(df_full_train['datetime']))

y_val = df_full_val[['count']]
y_val = y_val.set_index(pd.to_datetime(df_full_val['datetime']))

train_dataset, val_dataset = create_sequence_datasets(
    X_train, y_train, X_val, y_val,
    df_x_train['datetime'].values, df_x_val['datetime'].values
)

In [3]:
df_test = pd.read_csv('test.csv')

In [4]:
cat_features = ['season', 'weather', 'month', 'weekday', 'hour']
num_features = ['temp', 'atemp', 'humidity', 'windspeed']
# [holiday, workingday] ya son onehot
scaler = ColumnTransformer([('cat', OneHotEncoder(), cat_features),
                            ('num', StandardScaler(), num_features),
                            ], remainder='passthrough')

preprocess_pipe = Pipeline([('timefeatures', TimeFeatures()),
                            ('scaler', scaler)])

X_test = preprocess_pipe.fit_transform(df_test).toarray()

In [5]:
def create_test_sequence_dataset(X_test, sequence_length=24, batch_size=8):
    X_test_seq = []

    test_df = pd.DataFrame(X_test)

    test_df.index = pd.date_range(start='2011-01-01', periods=len(test_df), freq='H')

    for year in [2011, 2012]:
        for month in range(1, 13):
            start_date = datetime(year, month, 1)
            last_day_of_month = calendar.monthrange(year, month)[1]
            end_date = datetime(year, month, last_day_of_month, 23, 59, 59)

            df_month_test = test_df[(test_df.index >= start_date) & (test_df.index <= end_date)]

            if not df_month_test.empty:
                for i in range(len(df_month_test) - sequence_length):
                    X_test_seq.append(df_month_test.iloc[i:i+sequence_length].values)

    X_test_seq = np.array(X_test_seq)

    test_dataset = tf.data.Dataset.from_tensor_slices(X_test_seq).batch(batch_size)

    return test_dataset

test_dataset = create_test_sequence_dataset(X_test)

### Entrenar el mejor modelo

In [ ]:
model_seq_to_seq = keras.models.Sequential([
    keras.layers.Input(shape=(None, X_train.shape[-1])),
    keras.layers.SimpleRNN(64, return_sequences=True),
    keras.layers.TimeDistributed(keras.layers.Dense(1, activation='relu'))
])

optimizer = keras.optimizers.Adam(learning_rate=0.001)
model_seq_to_seq.compile(loss='mean_squared_logarithmic_error', metrics=['mean_absolute_error'], optimizer=optimizer)

history_seq_to_seq = model_seq_to_seq.fit(train_dataset, epochs=50, validation_data=val_dataset)

### Cargar modelo pre-entrenado

In [6]:
# Seq-to-seq con LSTM
model_seq_to_seq_lstm = keras.models.Sequential([
    keras.layers.Input(shape=(None, X_train.shape[-1])),
    keras.layers.LSTM(64, return_sequences=True),
    keras.layers.TimeDistributed(keras.layers.Dense(1, activation='relu'))
])


optimizer = keras.optimizers.Adam(learning_rate=0.001)
model_seq_to_seq_lstm.compile(loss='mean_squared_logarithmic_error',
                              metrics=['mean_absolute_error'],
                              optimizer=optimizer)

# Cargo pesos
model_seq_to_seq_lstm.load_weights('model_seq_to_seq_lstm.h5')

#### Predicciones del Modelo

In [7]:
predictions = model_seq_to_seq_lstm.predict(test_dataset)

print("Test:")
for x_batch in test_dataset.take(1):
    print(" Tamaño del batch (test):", x_batch.shape)


785/785 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step
Test:
 Tamaño del batch (test): (8, 24, 57)


2024-06-26 17:40:57.202896: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
